In [35]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import classification_report
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

In [36]:
df=pd.read_csv("Cleaned_Data.csv")

In [37]:
num_map = {
    'Extreme Fear': 1,
    'Fear': 2,
    'Neutral': 3,
    'Greed': 4,
    'Extreme Greed': 5
}
df['sentiment_score'] = df['classification'].map(num_map)

df = df.sort_values(['Account', 'date'])
df['next_day_pnl'] = df.groupby('Account')['daily_pnl'].shift(-1)
df_model = df.dropna(subset=['next_day_pnl']).copy()

df_model['target_bucket'] = pd.qcut(df_model['next_day_pnl'], q=3, labels=['Loss', 'Neutral', 'Profit'])

In [38]:
features = ['trades_per_day', 'avg_trade_size', 'win_rate', 'long_short_ratio', 'value', 'sentiment_score']
X = df_model[features]
y = df_model['target_bucket']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [39]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

In [40]:
param_grid = {
    'n_estimators': [100, 200, 300],     # Number of trees
    'max_depth': [None, 5, 10, 15],      # Maximum depth of the trees
    'min_samples_split': [2, 5, 10],     # Minimum samples required to split an internal node
    'min_samples_leaf': [1, 2, 4]        # Minimum samples required to be at a leaf node
}
# Set up the grid search with 5-fold cross-validation
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, scoring='f1_macro', n_jobs=-1)


In [41]:
grid_search.fit(X_train, y_train)
best_rf = grid_search.best_estimator_

y_pred_proba = best_rf.predict_proba(X_test)

print(y_pred_proba[:5])

[[0.3678876  0.23980606 0.39230634]
 [0.25143171 0.51041325 0.23815503]
 [0.5799966  0.14531815 0.27468525]
 [0.52849082 0.24935869 0.22215048]
 [0.35350591 0.27809419 0.3683999 ]]


In [42]:
print("Best Parameters:", grid_search.best_params_)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Best Parameters: {'max_depth': 5, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100}

Classification Report:

              precision    recall  f1-score   support

        Loss       0.54      0.60      0.57       174
     Neutral       0.53      0.47      0.49       129
      Profit       0.52      0.52      0.52       159

    accuracy                           0.53       462
   macro avg       0.53      0.53      0.53       462
weighted avg       0.53      0.53      0.53       462



In [43]:
trader_profiles = df.groupby('Account').agg({
    'trades_per_day': 'mean',
    'avg_trade_size': 'mean',
    'win_rate': 'mean',
    'long_short_ratio': 'mean'
}).dropna()

In [44]:
scaler = StandardScaler()
scaled_features = scaler.fit_transform(trader_profiles)

In [45]:
scaler = StandardScaler()
scaled_features = scaler.fit_transform(trader_profiles)

In [46]:
kmeans = KMeans(n_clusters=3, random_state=42)
trader_profiles['archetype'] = kmeans.fit_predict(scaled_features)

archetype_summary = trader_profiles.groupby('archetype').mean()
print("\nArchetype Summary:\n")
print(archetype_summary)


Archetype Summary:

           trades_per_day  avg_trade_size  win_rate  long_short_ratio
archetype                                                            
0              102.891885    16035.617479  0.417649         29.366198
1               83.794126     3833.798713  0.297870         21.604198
2              756.857143     4528.364243  0.455625        757.750000


In [47]:
import joblib

joblib.dump(rf, 'random_forest_model.pkl')
joblib.dump(kmeans, 'kmeans_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

['scaler.pkl']